In [2]:
import subprocess, requests, json, time

OLLAMA = "http://127.0.0.1:11434"
MODEL = "qwen2.5:7b-instruct-q4_K_M"
NOPROXY = {"http": None, "https": None}

# 1. GPU
print("=== GPU ===")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,memory.used,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout.strip())

# 2. Ollama server + model list
print("\n=== Ollama ===")
tags = requests.get(f"{OLLAMA}/api/tags", proxies=NOPROXY, timeout=10).json()
print("models:", [m["name"] for m in tags["models"]])

# 3. Qwen test generation + speed
print("\n=== Qwen test ===")
t0 = time.time()
r = requests.post(f"{OLLAMA}/api/chat", proxies=NOPROXY, timeout=60, json={
    "model": MODEL,
    "messages": [{"role": "user",
                  "content": "Paraphrase this sentence, output only the paraphrase: The cat sat on the mat."}],
    "stream": False,
    "options": {"temperature": 0.7}
}).json()
dt = time.time() - t0
print("output:", r["message"]["content"])
print(f"latency: {dt:.2f}s | tokens/s: {r['eval_count'] / (r['eval_duration'] / 1e9):.1f}")

# 4. Is the model actually on GPU?
print("\n=== Loaded on GPU? ===")
ps = requests.get(f"{OLLAMA}/api/ps", proxies=NOPROXY, timeout=10).json()
for m in ps.get("models", []):
    print(f"{m['name']}: {m['size_vram']/1e9:.1f} GB in VRAM of {m['size']/1e9:.1f} GB total")

=== GPU ===
NVIDIA A100-SXM4-80GB, 81920 MiB, 5617 MiB, 570.86.15

=== Ollama ===
models: ['qwen2.5:7b-instruct-q4_K_M']

=== Qwen test ===
output: The cat perched on the mat.
latency: 0.16s | tokens/s: 105.5

=== Loaded on GPU? ===
qwen2.5:7b-instruct-q4_K_M: 6.0 GB in VRAM of 6.0 GB total


In [4]:
import os, glob

print("cwd:", os.getcwd(), "\n")

for d in ["splits", "qwen_results", "qwen_export", "export"]:
    print(f"=== {d}/ ===")
    for f in sorted(glob.glob(f"{d}/**/*", recursive=True)):
        if os.path.isfile(f):
            print(f"{os.path.getsize(f)/1e6:8.2f} MB  {f}")
    print()

print("=== README.md (first 60 lines) ===")
with open("README.md") as fh:
    print("".join(fh.readlines()[:60]))

cwd: /home/bhandari.ru/NLP project 

=== splits/ ===
    0.09 MB  splits/mrpc_dev.csv
    0.10 MB  splits/mrpc_test.csv
    0.81 MB  splits/mrpc_train.csv
    1.87 MB  splits/paws_dev.csv
    1.88 MB  splits/paws_test.csv
   11.55 MB  splits/paws_train.csv
    0.50 MB  splits/qqp_dev.csv
    0.63 MB  splits/qqp_test.csv
    4.49 MB  splits/qqp_train.csv

=== qwen_results/ ===
    0.08 MB  qwen_results/figures/1_qwen_metrics_heatmap.png
    0.07 MB  qwen_results/figures/2_qwen_accuracy_vs_majority.png
    0.08 MB  qwen_results/figures/3_qwen_confusion.png
    0.07 MB  qwen_results/figures/4_qwen_prediction_bias.png
    0.08 MB  qwen_results/figures/5_qwen_per_class.png
    0.17 MB  qwen_results/figures/6_qwen_overlap_behavior.png
    0.00 MB  qwen_results/qwen_final_summary.csv
    0.00 MB  qwen_results/qwen_handwritten_results.csv
    0.00 MB  qwen_results/qwen_mrpc_preds.npy
    0.10 MB  qwen_results/qwen_mrpc_results.csv
    0.02 MB  qwen_results/qwen_paws_partial.csv
    0.06 MB  qw

In [5]:
import pandas as pd, os

for ds in ["paws", "qqp", "mrpc"]:
    for sp in ["train", "dev", "test"]:
        df = pd.read_csv(f"splits/{ds}_{sp}.csv")
        lab = df["label"].value_counts().to_dict() if "label" in df.columns else "no label col"
        print(f"{ds}_{sp:5s} rows={len(df):6d}  labels={lab}")

print("\ncolumns:", list(pd.read_csv("splits/paws_test.csv").columns))
print(pd.read_csv("splits/paws_test.csv").head(3).to_string(max_colwidth=70))

print("\n=== scratch folders ===")
for d in ["/scratch/bhandari.ru/paraphrase", "/scratch/bhandari.ru/paraphrase_llm"]:
    print(d, "->", sorted(os.listdir(d))[:20])

paws_train rows= 49259  labels={0: 27530, 1: 21729}
paws_dev   rows=  7983  labels={0: 4458, 1: 3525}
paws_test  rows=  8000  labels={0: 4464, 1: 3536}
qqp_train rows= 36000  labels={0: 22800, 1: 13200}
qqp_dev   rows=  4000  labels={0: 2569, 1: 1431}
qqp_test  rows=  5000  labels={0: 3165, 1: 1835}
mrpc_train rows=  3301  labels={1: 2237, 0: 1064}
mrpc_dev   rows=   367  labels={1: 237, 0: 130}
mrpc_test  rows=   408  labels={1: 279, 0: 129}

columns: ['sentence1', 'sentence2', 'label']
                                                               sentence1                                                              sentence2  label
0  This was a series of nested angular standards , so that measuremen...  This was a series of nested polar scales , so that measurements in...      0
1  His father emigrated to Missouri in 1868 but returned when his wif...  His father emigrated to America in 1868 , but returned when his wi...      0
2  In January 2011 , the Deputy Secretary General of F

In [6]:
import requests, time, pandas as pd

OLLAMA = "http://127.0.0.1:11434"
MODEL = "qwen2.5:7b-instruct-q4_K_M"
NOPROXY = {"http": None, "https": None}

SYSTEM = ("You are a precise paraphrasing assistant. Rewrite the given sentence using different "
          "wording and sentence structure while preserving the exact meaning. Keep all names, "
          "numbers, places, dates and relations unchanged. Do not add or remove information. "
          "Output only the paraphrase, nothing else.")

def paraphrase(sentence, temperature=0.7, seed=None):
    opts = {"temperature": temperature, "num_predict": 200}
    if seed is not None:
        opts["seed"] = seed
    r = requests.post(f"{OLLAMA}/api/chat", proxies=NOPROXY, timeout=60, json={
        "model": MODEL,
        "messages": [{"role": "system", "content": SYSTEM},
                     {"role": "user", "content": sentence}],
        "stream": False,
        "options": opts,
    }).json()
    return r["message"]["content"].strip()

# quick test on 5 PAWS test sentences
df = pd.read_csv("splits/paws_test.csv")
for i, row in df.head(5).iterrows():
    t0 = time.time()
    out = paraphrase(row.sentence1, seed=42)
    print(f"[{i}] label={row.label}  ({time.time()-t0:.2f}s)")
    print("  SRC :", row.sentence1)
    print("  GEN :", out)
    print("  REF :", row.sentence2, "\n")

[0] label=0  (2.44s)
  SRC : This was a series of nested angular standards , so that measurements in azimuth and elevation could be done directly in polar coordinates relative to the ecliptic .
  GEN : These standards were nested within the angular system, enabling direct measurements in polar coordinates for azimuth and elevation relative to the ecliptic.
  REF : This was a series of nested polar scales , so that measurements in azimuth and elevation could be performed directly in angular coordinates relative to the ecliptic . 

[1] label=0  (0.37s)
  SRC : His father emigrated to Missouri in 1868 but returned when his wife became ill and before the rest of the family could also go to America .
  GEN : In 1868, his father moved to Missouri, returning later due to his wife's illness, before the rest of the family could immigrate to America.
  REF : His father emigrated to America in 1868 , but returned when his wife became ill and before the rest of the family could go to Missouri . 



In [7]:
from concurrent.futures import ThreadPoolExecutor
import time

sample = df.sentence1.head(64).tolist()

for workers in [1, 8, 16]:
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=workers) as ex:
        outs = list(ex.map(lambda s: paraphrase(s, seed=42), sample))
    dt = time.time() - t0
    rate = len(sample) / dt
    print(f"workers={workers:2d}  {dt:5.1f}s  {rate:5.1f} sent/s  -> est. all 13.4k: {13408/rate/60:5.1f} min")

workers= 1   21.1s    3.0 sent/s  -> est. all 13.4k:  73.7 min
workers= 8   13.2s    4.9 sent/s  -> est. all 13.4k:  45.9 min
workers=16   13.1s    4.9 sent/s  -> est. all 13.4k:  45.8 min


In [1]:
import subprocess, os, requests

print("=== GPU ===")
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip())
except FileNotFoundError:
    print("nvidia-smi not found: this notebook is not on a GPU node")

try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
except Exception as e:
    print("torch check failed:", type(e).__name__)

print("\n=== Ollama ===")
try:
    r = requests.get("http://127.0.0.1:11434/api/tags", proxies={"http": None, "https": None}, timeout=5).json()
    print([m["name"] for m in r["models"]])
except Exception as e:
    print("not running:", type(e).__name__)

print("\n=== HF cache (models already downloaded) ===")
hf = "/scratch/bhandari.ru/hf_cache"
for root in [hf, os.path.join(hf, "hub")]:
    if os.path.isdir(root):
        print(root, "->", [d for d in sorted(os.listdir(root)) if d.startswith("models--")])

/home/bhandari.ru/.local/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.0.post1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


=== GPU ===
NVIDIA A100-SXM4-80GB, 81920 MiB, 27169 MiB
torch: 2.5.1+cu121 | cuda: True | NVIDIA A100-SXM4-80GB

=== Ollama ===
not running: ConnectionError

=== HF cache (models already downloaded) ===
/scratch/bhandari.ru/hf_cache -> []
/scratch/bhandari.ru/hf_cache/hub -> ['models--Qwen--Qwen2.5-7B-Instruct']


In [3]:
import os, glob

ROOTS = ["/home/bhandari.ru/NLP project", "/scratch/bhandari.ru/paraphrase"]
WEIGHT_PATTERNS = ["model.safetensors", "pytorch_model.bin", "*.pt", "*.pth", "*.ckpt"]
SKIP = ("ollama_work", "hf_cache", "apptainer", ".ipynb_checkpoints")

print("=== weight files ===")
found = []
for root in ROOTS:
    for pat in WEIGHT_PATTERNS:
        for f in glob.glob(os.path.join(root, "**", pat), recursive=True):
            if not any(s in f for s in SKIP):
                found.append(f)
for f in sorted(set(found)):
    print(f"{os.path.getsize(f)/1e6:9.1f} MB  {f}")
if not found:
    print("none found")

print("\n=== model folders (contain config.json) ===")
for root in ROOTS:
    for f in glob.glob(os.path.join(root, "**", "config.json"), recursive=True):
        if not any(s in f for s in SKIP):
            print(os.path.dirname(f))

print("\n=== scratch/paraphrase/out and out_team (top level) ===")
for d in ["out", "out_team"]:
    p = f"/scratch/bhandari.ru/paraphrase/{d}"
    print(d, "->", sorted(os.listdir(p))[:25])

=== weight files ===
    438.0 MB  /scratch/bhandari.ru/paraphrase/out/bert_paws_s42/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out/bert_qqp_s42/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_mrpc_s42_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_mrpc_s43_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_mrpc_s44_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_mrpc_s45_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_mrpc_s46_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_paws_s42_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_paws_s43_saved/best/model.safetensors
    438.0 MB  /scratch/bhandari.ru/paraphrase/out_team/bert_paws_s44_saved/best/model.safetensors
    438.0 MB  /scratch

In [4]:
import os, json, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BASE = "/scratch/bhandari.ru/paraphrase/out_team"
CK = f"{BASE}/bert_paws_s42_saved/best"
print("files in best/:", sorted(os.listdir(CK)))
print(json.dumps(json.load(open(f"{BASE}/bert_paws_s42_saved_metrics.json")), indent=1)[:500])

try:
    tok = AutoTokenizer.from_pretrained(CK); print("\ntokenizer: loaded from checkpoint folder")
except Exception as e:
    print("\nno tokenizer in folder:", type(e).__name__, "-> trying bert-base-uncased")
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModelForSequenceClassification.from_pretrained(CK).cuda().eval()

def score(s1, s2, bs=64, max_len=128):
    out = []
    for i in range(0, len(s1), bs):
        enc = tok(list(s1[i:i+bs]), list(s2[i:i+bs]), return_tensors="pt",
                  padding=True, truncation=True, max_length=max_len).to("cuda")
        with torch.no_grad():
            out += torch.softmax(model(**enc).logits, -1)[:, 1].cpu().tolist()
    return np.array(out)

# 1) two known pairs: first is label 0 (swapped), second is label 1
a = ["His father emigrated to Missouri in 1868 but returned when his wife became ill and before the rest of the family could also go to America .",
     "In January 2011 , the Deputy Secretary General of FIBA Asia , Hagop Khajirian , inspected the venue together with SBP - President Manuel V. Pangilinan ."]
b = ["His father emigrated to America in 1868 , but returned when his wife became ill and before the rest of the family could go to Missouri .",
     "In January 2011 , FIBA Asia deputy secretary general Hagop Khajirian along with SBP president Manuel V. Pangilinan inspected the venue ."]
print("\nP(paraphrase):", score(a, b).round(3), "(expect low, high)")

# 2) reproduce saved test predictions on the first 256 PAWS test rows
df = pd.read_csv("splits/paws_test.csv").head(256)
mine = (score(df.sentence1, df.sentence2) > 0.5).astype(int)
saved = np.load(f"{BASE}/bert_paws_s42_saved_test_preds.npy")[:256]
print("agreement with saved preds:", (mine == saved).mean(), "| accuracy vs labels:", (mine == df.label.values).mean())

files in best/: ['config.json', 'model.safetensors', 'training_args.bin']
{
 "tag": "bert_paws_s42_saved",
 "model": "bert-base-uncased",
 "dataset": "paws",
 "seed": 42,
 "n_train": 49259,
 "n_test": 8000,
 "test_accuracy": 0.907375,
 "test_precision": 0.8549657099314198,
 "test_recall": 0.9519230769230769,
 "test_f1": 0.9008430349257326,
 "majority_acc": 0.558,
 "frac_pred_paraphrase": 0.492125,
 "dev_curve": [
  {
   "epoch": 1.0,
   "dev_f1": 0.8908061472094905,
   "dev_loss": 0.26543551683425903
  },
  {
   "epoch": 2.0,
   "dev_f1": 0.907103825136612,
   "dev_lo

no tokenizer in folder: OSError -> trying bert-base-uncased


2026-11-21 13:19:04.487603: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795285144.500180 1593023 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795285144.504030 1593023 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795285144.515527 1593023 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795285144.515538 1593023 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795285144.515540 1593023 computation_placer.cc:177] computation placer alr


P(paraphrase): [0.002 0.998] (expect low, high)
agreement with saved preds: 1.0 | accuracy vs labels: 0.875


In [5]:
import glob, pandas as pd

for f in sorted(glob.glob("**/*.csv", recursive=True)):
    if "qwen" in f.lower():
        print(f, "->", list(pd.read_csv(f, nrows=1).columns))

print("\nnotebooks:", [n for n in glob.glob("**/*.ipynb", recursive=True)
                       if "checkpoint" not in n])

qwen_export/qwen_results/qwen_final_summary.csv -> ['Dataset', 'n', 'failures', 'Accuracy', 'F1']
qwen_export/qwen_results/qwen_handwritten_results.csv -> ['sentence1', 'sentence2', 'label', 'category', 'qwen_pred', 'correct']
qwen_export/qwen_results/qwen_mrpc_results.csv -> ['sentence1', 'sentence2', 'label', 'qwen_pred', 'pred']
qwen_export/qwen_results/qwen_paws_results.csv -> ['sentence1', 'sentence2', 'label', 'qwen_pred', 'pred']
qwen_export/qwen_results/qwen_qqp_results.csv -> ['sentence1', 'sentence2', 'label', 'qwen_pred', 'pred']
qwen_hf_results/mrpc_partial.csv -> ['qwen_logit_prob']
qwen_hf_results/paws_partial.csv -> ['qwen_logit_prob']
qwen_hf_results/qqp_partial.csv -> ['qwen_logit_prob']
qwen_hf_results/qwen_hf_mrpc_results.csv -> ['sentence1', 'sentence2', 'label', 'qwen_logit_prob', 'pred']
qwen_hf_results/qwen_hf_paws_results.csv -> ['sentence1', 'sentence2', 'label', 'qwen_logit_prob', 'pred']
qwen_hf_results/qwen_hf_qqp_results.csv -> ['sentence1', 'sentence2', 'l

In [6]:
import os, time, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_ID = "Qwen/Qwen2.5-7B-Instruct"
CACHE = "/scratch/bhandari.ru/hf_cache/hub"

gtok = AutoTokenizer.from_pretrained(HF_ID, cache_dir=CACHE, local_files_only=True, padding_side="left")
gen_model = AutoModelForCausalLM.from_pretrained(
    HF_ID, cache_dir=CACHE, local_files_only=True,
    torch_dtype=torch.bfloat16, device_map="cuda").eval()

SYSTEM = ("You are a precise paraphrasing assistant. Rewrite the given sentence using different "
          "wording and sentence structure while preserving the exact meaning. Keep all names, "
          "numbers, places, dates and relations unchanged. Do not add or remove information. "
          "Output only the paraphrase, nothing else.")

def paraphrase_batch(sentences, temperature=0.7, max_new_tokens=128):
    prompts = [gtok.apply_chat_template(
                   [{"role": "system", "content": SYSTEM}, {"role": "user", "content": s}],
                   tokenize=False, add_generation_prompt=True) for s in sentences]
    enc = gtok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = gen_model.generate(**enc, do_sample=True, temperature=temperature, top_p=0.9, top_k=40,
                                 max_new_tokens=max_new_tokens, pad_token_id=gtok.pad_token_id)
    new = out[:, enc.input_ids.shape[1]:]
    texts = gtok.batch_decode(new, skip_special_tokens=True)
    # flag outputs that hit the token limit (possibly cut off)
    hit_limit = [(row != gtok.pad_token_id).sum().item() >= max_new_tokens for row in new]
    return [t.strip() for t in texts], hit_limit

torch.manual_seed(42)
df = pd.read_csv("splits/paws_test.csv")
sample = df.sentence1.head(64).tolist()

paraphrase_batch(sample[:2])                         # warm-up, not timed
t0 = time.time(); outs, cut = paraphrase_batch(sample); dt = time.time() - t0

print(f"64 sentences in {dt:.1f}s -> {64/dt:.1f} sent/s | est. all 13,408: {13408/(64/dt)/60:.1f} min")
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB | truncated outputs: {sum(cut)}")
for s, o in list(zip(sample, outs))[:5]:
    print("\nSRC:", s)
    print("GEN:", o)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

64 sentences in 2.6s -> 25.0 sent/s | est. all 13,408: 9.0 min
GPU memory allocated: 15.7 GB | truncated outputs: 0

SRC: This was a series of nested angular standards , so that measurements in azimuth and elevation could be done directly in polar coordinates relative to the ecliptic .
GEN: These standards were nested within the angular system, allowing for direct measurements in polar coordinates regarding the ecliptic in terms of azimuth and elevation.

SRC: His father emigrated to Missouri in 1868 but returned when his wife became ill and before the rest of the family could also go to America .
GEN: In 1868, his father moved to Missouri, returning there due to his wife's illness before the rest of the family could immigrate to America.

SRC: In January 2011 , the Deputy Secretary General of FIBA Asia , Hagop Khajirian , inspected the venue together with SBP - President Manuel V. Pangilinan .
GEN: In January 2011, Hagop Khajirian, the Deputy Secretary General of FIBA Asia, jointly ex

In [7]:
import os, json, time, datetime, torch, transformers, pandas as pd

OUT = "qwen_hf_generation/taskA"
os.makedirs(OUT, exist_ok=True)
SEED, BS, SAVE_EVERY = 42, 128, 5          # save progress every 5 batches (640 sentences)
GEN_CFG = dict(temperature=0.7, top_p=0.9, top_k=40, max_new_tokens=128)

def run_taskA(name):
    final = f"{OUT}/{name}_taskA.csv"
    if os.path.exists(final):
        print(f"{name}: already done, skipping")
        return pd.read_csv(final)

    df = pd.read_csv(f"splits/{name}_test.csv").reset_index().rename(columns={"index": "idx"})
    df["sentence1"] = df["sentence1"].astype(str)
    part = f"{OUT}/{name}_partial.csv"
    done = pd.read_csv(part) if os.path.exists(part) else pd.DataFrame(columns=["idx", "gen", "truncated"])

    # sort by length so each batch has similar lengths (less padding, faster)
    todo = df[~df.idx.isin(done.idx)]
    todo = todo.assign(_len=todo.sentence1.str.len()).sort_values(["_len", "idx"])
    b0 = len(done) // BS                   # batch counter continues after a resume
    print(f"{name}: {len(df)} rows | {len(done)} already done | {len(todo)} to go")

    rows, t0 = [], time.time()
    for i in range(0, len(todo), BS):
        chunk = todo.iloc[i:i+BS]
        torch.manual_seed(SEED + b0 + i // BS)   # fixed seed per batch for reproducibility
        gens, cut = paraphrase_batch(chunk.sentence1.tolist(), **{k: GEN_CFG[k] for k in ["temperature", "max_new_tokens"]})
        rows += [{"idx": j, "gen": g, "truncated": c} for j, g, c in zip(chunk.idx, gens, cut)]

        n_batches = i // BS + 1
        if n_batches % SAVE_EVERY == 0 or i + BS >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); rows = []
            done.to_csv(part, index=False)
            rate = (i + len(chunk)) / (time.time() - t0)
            print(f"  {len(done):5d}/{len(df)}  {rate:5.1f} sent/s  ETA {(len(df)-len(done))/rate/60:4.1f} min")

    res = df.merge(done, on="idx", how="left").sort_values("idx")
    assert len(res) == len(df) and res.gen.notna().all(), f"{name}: missing generations"
    res = res[["idx", "sentence1", "sentence2", "label", "gen", "truncated"]]
    res.to_csv(final, index=False)
    os.remove(part)
    print(f"{name}: saved {final}")
    return res

results = {n: run_taskA(n) for n in ["mrpc", "qqp", "paws"]}

# record exactly how this was generated
meta = {"task": "A: paraphrase sentence1 of every test row",
        "model": HF_ID, "dtype": "bfloat16", "backend": "transformers " + transformers.__version__,
        "system_prompt": SYSTEM, "gen_config": GEN_CFG, "seed": SEED, "batch_size": BS,
        "date": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": {n: len(r) for n, r in results.items()}}
json.dump(meta, open(f"{OUT}/taskA_metadata.json", "w"), indent=1)

# quick health check
for n, r in results.items():
    g, s = r.gen.str.strip(), r.sentence1.str.strip()
    print(f"{n:5s} n={len(r):5d} | truncated={r.truncated.sum():3d} | empty={(g == '').sum():3d} | "
          f"exact copy={(g.str.lower() == s.str.lower()).sum():3d} | "
          f"len ratio gen/src={(g.str.len() / s.str.len()).median():.2f}")

mrpc: 408 rows | 0 already done | 408 to go
    408/408   32.2 sent/s  ETA  0.0 min
mrpc: saved qwen_hf_generation/taskA/mrpc_taskA.csv
qqp: 5000 rows | 0 already done | 5000 to go
    640/5000   76.0 sent/s  ETA  1.0 min
   1280/5000   79.9 sent/s  ETA  0.8 min
   1920/5000   75.9 sent/s  ETA  0.7 min
   2560/5000   75.4 sent/s  ETA  0.5 min
   3200/5000   73.5 sent/s  ETA  0.4 min
   3840/5000   68.5 sent/s  ETA  0.3 min
   4480/5000   64.4 sent/s  ETA  0.1 min
   5000/5000   59.1 sent/s  ETA  0.0 min
qqp: saved qwen_hf_generation/taskA/qqp_taskA.csv
paws: 8000 rows | 0 already done | 8000 to go
    640/8000   62.5 sent/s  ETA  2.0 min
   1280/8000   53.6 sent/s  ETA  2.1 min
   1920/8000   52.8 sent/s  ETA  1.9 min
   2560/8000   51.1 sent/s  ETA  1.8 min
   3200/8000   47.3 sent/s  ETA  1.7 min
   3840/8000   46.2 sent/s  ETA  1.5 min
   4480/8000   44.8 sent/s  ETA  1.3 min
   5120/8000   42.2 sent/s  ETA  1.1 min
   5760/8000   41.4 sent/s  ETA  0.9 min
   6400/8000   40.6 sent/s

In [8]:
q = results["qqp"].copy()
q["ratio"] = q.gen.str.len() / q.sentence1.str.len()
src_q = q.sentence1.str.strip().str.endswith("?")
gen_q = q.gen.str.strip().str.endswith("?")

print("ratio distribution:", q.ratio.describe(percentiles=[.5, .9, .99]).round(2).to_dict())
print(f"ratio > 2: {(q.ratio > 2).sum()} | ratio > 3: {(q.ratio > 3).sum()}")
print(f"source is a question but GEN is not: {(src_q & ~gen_q).sum()} of {src_q.sum()}")

print("\n=== 8 longest relative rewrites ===")
for _, r in q.sort_values("ratio", ascending=False).head(8).iterrows():
    print(f"\n[{r.idx}] ratio={r.ratio:.1f}")
    print("  SRC:", r.sentence1)
    print("  GEN:", r.gen[:300])

print("\n=== 5 random 'question -> non-question' cases ===")
for _, r in q[src_q & ~gen_q].sample(5, random_state=0).iterrows():
    print(f"\n[{r.idx}]  SRC: {r.sentence1}\n        GEN: {r.gen[:300]}")

ratio distribution: {'count': 5000.0, 'mean': 1.29, 'std': 0.3, 'min': 0.5, '50%': 1.25, '90%': 1.67, '99%': 2.23, 'max': 4.0}
ratio > 2: 120 | ratio > 3: 3
source is a question but GEN is not: 35 of 4843

=== 8 longest relative rewrites ===

[866] ratio=4.0
  SRC: What?
  GEN: Can you repeat that?

[1382] ratio=4.0
  SRC: Who is the next MJ?
  GEN: What artist will be the next one to achieve the same level of success as MJ?

[2282] ratio=4.0
  SRC: Why does everyone make fun of Nicolas Cage?
  GEN: For what reason do people常規表达方式： Why does everyone 常规表达方式： why is it that everyone 常规表达方式： why do so many people  Everyone making fun of Nicolas Cage, can you explain why?

[1369] ratio=2.9
  SRC: How do I grow hair close?
  GEN: What is the method for encouraging hair growth in a short, close manner?

[328] ratio=2.7
  SRC: Dating a married man?
  GEN: Is it appropriate to date someone who is already married?

[2020] ratio=2.7
  SRC: Can Donald Trump win?
  GEN: Is it possible for Donald T

In [9]:
import re

CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")   # Chinese/Japanese/Korean + full-width
META = re.compile(r"^(here|paraphrase|rewritten|sure|certainly)\b|paraphrase:", re.I)

for n, r in results.items():
    g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
    cjk = g.str.contains(CJK)
    non_ascii = g.str.contains(r"[^\x00-\x7F]") & ~s.str.contains(r"[^\x00-\x7F]")
    q2s = s.str.endswith("?") & ~g.str.endswith("?")
    meta = g.str.contains(META)
    bad = cjk | q2s | meta
    print(f"{n:5s} n={len(r):5d} | CJK={cjk.sum():3d} | new non-ASCII={non_ascii.sum():3d} | "
          f"question->statement={q2s.sum():3d} | meta text={meta.sum():3d} | total flagged={bad.sum():3d} ({bad.mean():.1%})")

print("\n=== sample of non-CJK non-ASCII cases (check these are fine, e.g. accents) ===")
allr = pd.concat([r.assign(ds=n) for n, r in results.items()])
g = allr.gen.astype(str)
mask = g.str.contains(r"[^\x00-\x7F]") & ~g.str.contains(CJK) & ~allr.sentence1.astype(str).str.contains(r"[^\x00-\x7F]")
for _, r in allr[mask].head(5).iterrows():
    print(f"[{r.ds} {r.idx}] {r.gen[:200]}")

mrpc  n=  408 | CJK=  4 | new non-ASCII=  6 | question->statement=  0 | meta text=  0 | total flagged=  4 (1.0%)
qqp   n= 5000 | CJK= 22 | new non-ASCII= 18 | question->statement= 35 | meta text=  0 | total flagged= 50 (1.0%)
paws  n= 8000 | CJK= 38 | new non-ASCII= 68 | question->statement=  0 | meta text= 14 | total flagged= 52 (0.7%)

=== sample of non-CJK non-ASCII cases (check these are fine, e.g. accents) ===
[mrpc 104] Parrish stated, "I’m unsure if the person I’m speaking to could potentially be different and possibly not adhere to the rules at some other point in time."
[mrpc 173] On November 17, the most recent victim was British defence attaché Stephen Saunders, who was shot on an Athens road in June 2000.
[mrpc 257] Europe 1 radio reported that metro, bus, and local rail services in France's four biggest cities—Paris, Lyon, Lille, and Marseille—experienced significant disruptions.
[qqp 2289] What is the method to capture a Charizard in Pokémon Go?
[qqp 3070] (1+1+1+1+1+1+1+

/tmp/ipykernel_1593023/364119544.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  meta = g.str.contains(META)
/tmp/ipykernel_1593023/364119544.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  meta = g.str.contains(META)
/tmp/ipykernel_1593023/364119544.py:11: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  meta = g.str.contains(META)


In [1]:
import os, re, json, shutil, torch, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

HF_ID = "Qwen/Qwen2.5-7B-Instruct"
CACHE = "/scratch/bhandari.ru/hf_cache/hub"
V1 = "qwen_hf_generation/taskA_v1_unfixed"

# --- 1) keep v1: copy the unfixed run to its own folder ---
if not os.path.exists(V1):
    shutil.copytree("qwen_hf_generation/taskA", V1)
results = {n: pd.read_csv(f"{V1}/{n}_taskA.csv") for n in ["mrpc", "qqp", "paws"]}
print("v1 loaded:", {n: len(r) for n, r in results.items()})

# --- 2) load Qwen only if it's not already in memory ---
if "gen_model" not in globals():
    gtok = AutoTokenizer.from_pretrained(HF_ID, cache_dir=CACHE, local_files_only=True, padding_side="left")
    gen_model = AutoModelForCausalLM.from_pretrained(HF_ID, cache_dir=CACHE, local_files_only=True,
                                                     torch_dtype=torch.bfloat16, device_map="cuda").eval()
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.1f} GB")

# --- 3) failure checks; flag every v1 row and save them for the report ---
CJK  = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")
META = re.compile(r"^(?:here|paraphrase|rewritten|sure|certainly)\b|paraphrase:", re.I)

def flags(src, gen):
    src, gen = src.astype(str).str.strip(), gen.astype(str).str.strip()
    return pd.DataFrame({"cjk": gen.str.contains(CJK).values,
                         "q2s": (src.str.endswith("?") & ~gen.str.endswith("?")).values,
                         "meta": gen.str.contains(META).values})

flagged = []
for n, r in results.items():
    f = flags(r.sentence1, r.gen)
    flagged.append(pd.concat([r.reset_index(drop=True), f], axis=1)[f.any(axis=1).values].assign(dataset=n))
flagged = pd.concat(flagged, ignore_index=True)
flagged.to_csv(f"{V1}/v1_flagged_rows.csv", index=False)
print(flagged.groupby("dataset")[["cjk", "q2s", "meta"]].sum())

# --- 4) inspect the 14 PAWS "meta" rows: real problem or false positive? ---
print("\n=== PAWS meta rows ===")
for _, r in flagged[(flagged.dataset == "paws") & flagged.meta].head(6).iterrows():
    print(f"[{r.idx}] SRC: {r.sentence1[:110]}\n        GEN: {r.gen[:150]}")

# --- 5) the fix: clearer prompt + block CJK tokens ---
special = set(gtok.all_special_ids)
SUPPRESS = [i for i in range(len(gtok)) if i not in special and CJK.search(gtok.decode([i]))]
print(f"\nblocked {len(SUPPRESS)} of {len(gtok)} vocabulary tokens")

SYSTEM_V2 = ("You are a precise paraphrasing assistant. Rewrite the given sentence using different "
             "wording and sentence structure while preserving the exact meaning. Keep all names, "
             "numbers, places, dates and relations unchanged. Do not add or remove information. "
             "If the input is a question, the paraphrase must also be a question; do not answer it. "
             "Respond in English only. Output only the paraphrase, nothing else.")

def paraphrase_batch_v2(sentences, temperature=0.7, max_new_tokens=128):
    prompts = [gtok.apply_chat_template([{"role": "system", "content": SYSTEM_V2},
                                         {"role": "user", "content": s}],
                                        tokenize=False, add_generation_prompt=True) for s in sentences]
    enc = gtok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = gen_model.generate(**enc, do_sample=True, temperature=temperature, top_p=0.9, top_k=40,
                                 max_new_tokens=max_new_tokens, pad_token_id=gtok.pad_token_id,
                                 suppress_tokens=SUPPRESS)
    new = out[:, enc.input_ids.shape[1]:]
    hit = [(row != gtok.pad_token_id).sum().item() >= max_new_tokens for row in new]
    return [t.strip() for t in gtok.batch_decode(new, skip_special_tokens=True)], hit

# --- 6) test the fix on the flagged rows only ---
torch.manual_seed(42)
new, _ = paraphrase_batch_v2(flagged.sentence1.tolist())
f2 = flags(flagged.sentence1, pd.Series(new))
print("\nafter fix, still flagged:", f2.sum().to_dict(), f"(was {flagged[['cjk','q2s','meta']].sum().to_dict()})")

print("\n=== examples: old -> new ===")
for i in flagged.index[flagged.idx.isin([2612, 4006, 2042, 2282, 2466]) & (flagged.dataset == "qqp")]:
    print(f"[{flagged.idx[i]}] SRC: {flagged.sentence1[i]}\n      OLD: {flagged.gen[i][:110]}\n      NEW: {new[i][:110]}\n")

/home/bhandari.ru/.local/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.0.post1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


v1 loaded: {'mrpc': 408, 'qqp': 5000, 'paws': 8000}


2026-11-24 01:48:24.202824: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795502904.427614  517744 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795502904.494597  517744 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795502905.157154  517744 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795502905.157188  517744 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795502905.157190  517744 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

GPU memory: 15.3 GB
         cjk  q2s  meta
dataset                
mrpc       4    0     0
paws      38    0    14
qqp       22   35     0

=== PAWS meta rows ===
[346] SRC: This is a list of international sporting events in the Greater Manila Area in the Philippines :
        GEN: Here is a list of international sports events held in the Greater Manila Area of the Philippines:
[1574] SRC: This is a list of international sporting events in the Greater Manila Area in the Philippines :
        GEN: Here is a list of international sports events held in the Greater Manila Area of the Philippines:
[2325] SRC: This is a list of international sporting events in the Philippines in the Greater Manila Area :
        GEN: Here is a list of international sports events held in the Greater Manila Area of the Philippines:
[2470] SRC: This is a list of the international sporting events in the Greater Manila Area in the Philippines :
        GEN: Here is a list of international sporting events held in

In [2]:
import os, json, time, datetime, transformers, torch, pandas as pd

OUT = "qwen_hf_generation/taskA_v2"
os.makedirs(OUT, exist_ok=True)
SEED, BS, SAVE_EVERY, MAX_RETRY = 42, 128, 5, 3
GEN_CFG = dict(temperature=0.7, top_p=0.9, top_k=40, max_new_tokens=128)

# look at the 3 leftovers from the test first
left = flagged.assign(new=new)[(f2.cjk | f2.q2s).values]
for _, r in left.iterrows():
    print(f"[{r.dataset} {r.idx}] SRC: {r.sentence1[:120]}\n      NEW: {r.new[:120]}\n")

def bad_mask(src, gen):
    f = flags(src, gen)
    return (f.cjk | f.q2s).values      # meta check dropped (false positives)

def run_v2(name):
    final = f"{OUT}/{name}_taskA.csv"
    if os.path.exists(final):
        print(f"{name}: already done, skipping"); return pd.read_csv(final)

    df = pd.read_csv(f"splits/{name}_test.csv").reset_index().rename(columns={"index": "idx"})
    df["sentence1"] = df["sentence1"].astype(str)
    part = f"{OUT}/{name}_partial.csv"
    done = pd.read_csv(part) if os.path.exists(part) else pd.DataFrame(columns=["idx", "gen", "truncated"])
    todo = df[~df.idx.isin(done.idx)]
    todo = todo.assign(_len=todo.sentence1.str.len()).sort_values(["_len", "idx"])
    b0 = len(done) // BS
    print(f"{name}: {len(df)} rows | {len(done)} done | {len(todo)} to go")

    rows, t0 = [], time.time()
    for i in range(0, len(todo), BS):
        chunk = todo.iloc[i:i+BS]
        torch.manual_seed(SEED + b0 + i // BS)
        gens, cut = paraphrase_batch_v2(chunk.sentence1.tolist(), GEN_CFG["temperature"], GEN_CFG["max_new_tokens"])
        rows += [{"idx": j, "gen": g, "truncated": c} for j, g, c in zip(chunk.idx, gens, cut)]
        if (i // BS + 1) % SAVE_EVERY == 0 or i + BS >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); rows = []
            done.to_csv(part, index=False)
            rate = (i + len(chunk)) / (time.time() - t0)
            print(f"  {len(done):5d}/{len(df)}  {rate:5.1f} sent/s  ETA {(len(df)-len(done))/rate/60:4.1f} min")

    res = df.merge(done, on="idx", how="left").sort_values("idx").reset_index(drop=True)
    assert len(res) == len(df) and res.gen.notna().all(), f"{name}: missing generations"

    # retry rows that still fail, with a new seed each attempt
    res["n_retries"] = 0
    for attempt in range(1, MAX_RETRY + 1):
        bad = bad_mask(res.sentence1, res.gen)
        if not bad.any(): break
        torch.manual_seed(SEED + 100_000 + attempt)
        regen, cut = paraphrase_batch_v2(res.sentence1[bad].tolist(), GEN_CFG["temperature"], GEN_CFG["max_new_tokens"])
        res.loc[bad, "gen"], res.loc[bad, "truncated"] = regen, cut
        res.loc[bad, "n_retries"] = attempt
        print(f"  retry {attempt}: regenerated {bad.sum()} rows")

    f = flags(res.sentence1, res.gen)
    res["flag_cjk"], res["flag_q2s"] = f.cjk.values, f.q2s.values   # anything left after 3 retries stays visible
    res = res[["idx", "sentence1", "sentence2", "label", "gen", "truncated", "n_retries", "flag_cjk", "flag_q2s"]]
    res.to_csv(final, index=False); os.remove(part)
    print(f"{name}: saved {final}")
    return res

results_v2 = {n: run_v2(n) for n in ["mrpc", "qqp", "paws"]}

meta = {"task": "A v2: paraphrase sentence1 of every test row (fixed prompt + CJK token block)",
        "model": HF_ID, "dtype": "bfloat16", "backend": "transformers " + transformers.__version__,
        "system_prompt": SYSTEM_V2, "gen_config": GEN_CFG, "seed": SEED, "batch_size": BS,
        "suppressed_tokens": len(SUPPRESS), "retry_policy": f"up to {MAX_RETRY} retries for CJK or question->statement",
        "date": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": {n: len(r) for n, r in results_v2.items()}}
json.dump(meta, open(f"{OUT}/taskA_v2_metadata.json", "w"), indent=1)

print()
for n, r in results_v2.items():
    g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
    print(f"{n:5s} n={len(r):5d} | truncated={r.truncated.sum():2d} | empty={(g == '').sum():2d} | "
          f"copies={(g.str.lower() == s.str.lower()).sum():2d} | retried={(r.n_retries > 0).sum():3d} | "
          f"still CJK={r.flag_cjk.sum()} | still q->s={r.flag_q2s.sum()} | len ratio={(g.str.len()/s.str.len()).median():.2f}")

[qqp 839] SRC: What are some nerdy pick up lines?
      NEW: Can I haz your phone number? I heard you're a codebreaker.

[qqp 1587] SRC: Are East Asian people (i.e, Koreans, Chinese, Thai, etc) considered a 外国人 gaikokujin in Japan? Are they treated worse or
      NEW: Do East Asian individuals such as Koreans, Chinese, and Thais (etc.) count as 外 foreigners in Japan? Are they treated mo

[qqp 4747] SRC: I moved to a new school and nobody talks to me and I don’t know what to talk about with them?
      NEW: When I transferred to a new school, no one is speaking to me, and I'm unsure of topics to discuss with them.

mrpc: 408 rows | 0 done | 408 to go
    408/408   28.7 sent/s  ETA  0.0 min
mrpc: saved qwen_hf_generation/taskA_v2/mrpc_taskA.csv
qqp: 5000 rows | 0 done | 5000 to go
    640/5000   59.2 sent/s  ETA  1.2 min
   1280/5000   62.5 sent/s  ETA  1.0 min
   1920/5000   62.7 sent/s  ETA  0.8 min
   2560/5000   61.2 sent/s  ETA  0.7 min
   3200/5000   59.5 sent/s  ETA  0.5 min
   38

In [3]:
CJK_SRC = lambda s: s.astype(str).str.contains(CJK)

for n, r in results_v2.items():
    r["flag_cjk"] = r.flag_cjk & ~CJK_SRC(r.sentence1)            # source already had CJK -> not leakage
    left = r[r.flag_cjk | r.flag_q2s]
    for _, x in left.iterrows():
        kind = "CJK" if x.flag_cjk else "q->s"
        print(f"[{n} {x.idx}] {kind} | retries={x.n_retries}\n   SRC: {x.sentence1[:140]}\n   GEN: {x.gen[:140]}\n")
    r.to_csv(f"qwen_hf_generation/taskA_v2/{n}_taskA.csv", index=False)

print("remaining flagged:", {n: int((r.flag_cjk | r.flag_q2s).sum()) for n, r in results_v2.items()})

[qqp 4747] q->s | retries=3
   SRC: I moved to a new school and nobody talks to me and I don’t know what to talk about with them?
   GEN: When I transferred to a new school, no one approaches me, and I'm unsure of topics to discuss with them.

remaining flagged: {'mrpc': 0, 'qqp': 1, 'paws': 0}


In [13]:
# ===== Evaluation helpers (nothing is computed here) =====
import sys, subprocess, re, os, numpy as np, pandas as pd
try:
    import sacrebleu
    from bert_score import BERTScorer
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "-q", "sacrebleu", "bert-score"])
    import sacrebleu
    from bert_score import BERTScorer

DATASETS     = ["mrpc", "qqp", "paws"]
GEN_UNFIXED  = "qwen_hf_generation/taskA_v1_unfixed"     # original prompt
GEN_FIXED    = "qwen_hf_generation/taskA_v2"             # fixed prompt + CJK block + retries
METRICS_DIR  = "qwen_hf_generation/metrics"
VS_SOURCE_DIR = f"{METRICS_DIR}/vs_source"
VS_HUMAN_DIR  = f"{METRICS_DIR}/vs_human_reference"
for d in (VS_SOURCE_DIR, VS_HUMAN_DIR):
    os.makedirs(d, exist_ok=True)
IBLEU_ALPHA = 0.8   # iBLEU = 0.8*BLEU(gen, human) - 0.2*BLEU(gen, source)

def clean_text(t):
    """Same cleanup for source, human reference and generation, so formatting differences don't count."""
    t = str(t)
    t = (t.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
          .replace("\u2014", " - ").replace("\u2013", "-"))
    t = re.sub(r"\s+([,.;:!?%)\]])", r"\1", t)          # "2011 ," -> "2011,"
    t = re.sub(r"([(\[])\s+", r"\1", t)                   # "( born" -> "(born"
    t = re.sub(r"\s+'(s|re|ve|ll|d|m)\b", r"'\1", t)       # "John 's" -> "John's"
    t = re.sub(r"\s+n't\b", "n't", t)
    return re.sub(r"\s+", " ", t).strip()

def sentence_bleu_scores(hyps, refs):
    return np.array([sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hyps, refs)])

def corpus_bleu_score(hyps, refs):
    return sacrebleu.corpus_bleu(list(hyps), [list(refs)]).score

def bertscore_f1(cands, refs):
    """BERTScore F1 (roberta-large, rescaled with baseline); the scorer loads once on first use."""
    g = globals()
    if g.get("_bertscorer") is None:
        g["_bertscorer"] = BERTScorer(lang="en", rescale_with_baseline=True, batch_size=128, device="cuda")
    return g["_bertscorer"].score(list(cands), list(refs))[2].numpy()

def bootstrap_ci95(x, n_boot=1000, seed=0):
    x = np.asarray(x, float)
    means = np.random.default_rng(seed).choice(x, (n_boot, len(x))).mean(1)
    return np.percentile(means, [2.5, 97.5]).round(3)

def load_generated_paraphrases(dataset, folder=GEN_FIXED):
    return pd.read_csv(f"{folder}/{dataset}_taskA.csv")

def load_cached_scores(dataset, generated):
    """Per-row BLEU/BERTScore already computed earlier, if the file exists and rows line up."""
    path = f"{METRICS_DIR}/{dataset}_auto_metrics.csv"
    if not os.path.exists(path):
        return None
    cached = pd.read_csv(path)
    aligned = len(cached) == len(generated) and (cached.idx.values == generated.idx.values).all()
    return cached if aligned else None

print("evaluation helpers ready")

evaluation helpers ready


In [6]:
# ===== Generation failures: unfixed vs fixed prompt =====
CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")

def detect_generation_failures(sources, generations):
    src = pd.Series(list(sources)).astype(str).str.strip()
    gen = pd.Series(list(generations)).astype(str).str.strip()
    return pd.DataFrame({"chinese_text": gen.str.contains(CJK) & ~src.str.contains(CJK),   # source had none
                         "question_to_statement": src.str.endswith("?") & ~gen.str.endswith("?")})

def compare_unfixed_vs_fixed():
    rows = []
    for version, folder in [("unfixed", GEN_UNFIXED), ("fixed", GEN_FIXED)]:
        for ds in DATASETS:
            r = load_generated_paraphrases(ds, folder)
            f = detect_generation_failures(r.sentence1, r.gen)
            g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
            rows.append({"version": version, "dataset": ds, "n": len(r),
                         "chinese_text": int(f.chinese_text.sum()),
                         "question_to_statement": int(f.question_to_statement.sum()),
                         "flagged_%": round(100 * f.any(axis=1).mean(), 2),
                         "empty": int((g == "").sum()),
                         "copies": int((g.str.lower() == s.str.lower()).sum()),
                         "len_ratio_median": round((g.str.len() / s.str.len()).median(), 2)})
    return pd.DataFrame(rows)

failures = compare_unfixed_vs_fixed()
failures.to_csv(f"{METRICS_DIR}/generation_failures_unfixed_vs_fixed.csv", index=False)
print(failures.to_string(index=False))

version dataset    n  chinese_text  question_to_statement  flagged_%  empty  copies  len_ratio_median
unfixed    mrpc  408             4                      0       0.98      0       0              1.06
unfixed     qqp 5000            16                     35       0.88      0       0              1.25
unfixed    paws 8000            36                      0       0.45      0       0              1.04
  fixed    mrpc  408             0                      0       0.00      0       0              1.07
  fixed     qqp 5000             0                      1       0.02      0       0              1.23
  fixed    paws 8000             0                      0       0.00      0       0              1.06


In [7]:
# ===== Paraphrase vs source: meaning kept (BERTScore) and rewording (BLEU), every test row =====
def score_paraphrases_vs_source(dataset):
    generated = load_generated_paraphrases(dataset)
    r = generated[["idx", "sentence1", "gen", "label"]].copy()
    src, gen = r.sentence1.map(clean_text), r.gen.map(clean_text)

    cached = load_cached_scores(dataset, generated)
    if cached is not None:
        r["bleu_gen_src"], r["bertscore_gen_src"] = cached.bleu_gen_src.values, cached.bs_gen_src.values
    else:
        r["bleu_gen_src"], r["bertscore_gen_src"] = sentence_bleu_scores(gen, src), bertscore_f1(gen, src)
    r.to_csv(f"{VS_SOURCE_DIR}/{dataset}_scores.csv", index=False)

    lo, hi = bootstrap_ci95(r.bertscore_gen_src)
    return {"dataset": dataset, "n": len(r),
            "BERTScore_gen_src": round(r.bertscore_gen_src.mean(), 3), "BERTScore_CI95": f"[{lo}, {hi}]",
            "BLEU_gen_src": round(corpus_bleu_score(gen, src), 2)}

vs_source = pd.DataFrame([score_paraphrases_vs_source(ds) for ds in DATASETS])
vs_source.to_csv(f"{VS_SOURCE_DIR}/summary.csv", index=False)
print(vs_source.to_string(index=False))

dataset    n  BERTScore_gen_src BERTScore_CI95  BLEU_gen_src
   mrpc  408              0.676 [0.666, 0.685]         28.18
    qqp 5000              0.657  [0.654, 0.66]         18.90
   paws 8000              0.667  [0.665, 0.67]         33.01


In [8]:
# ===== Paraphrase vs human reference: label-1 rows, Qwen's paraphrase vs the human sentence2 =====
def score_paraphrases_vs_human(dataset):
    generated = load_generated_paraphrases(dataset)
    is_para = (generated.label == 1).values
    r = generated.loc[is_para, ["idx", "sentence1", "sentence2", "gen"]].reset_index(drop=True)
    src, human, gen = r.sentence1.map(clean_text), r.sentence2.map(clean_text), r.gen.map(clean_text)

    cached = load_cached_scores(dataset, generated)
    if cached is not None:
        c = cached[is_para].reset_index(drop=True)
        r["bleu_gen_src"], r["bleu_human_src"], r["bleu_gen_human"] = c.bleu_gen_src, c.bleu_ref_src, c.bleu_gen_ref
        r["bertscore_gen_src"], r["bertscore_human_src"], r["bertscore_gen_human"] = c.bs_gen_src, c.bs_ref_src, c.bs_gen_ref
    else:
        r["bleu_gen_src"]   = sentence_bleu_scores(gen, src)
        r["bleu_human_src"] = sentence_bleu_scores(human, src)
        r["bleu_gen_human"] = sentence_bleu_scores(gen, human)
        r["bertscore_gen_src"]   = bertscore_f1(gen, src)
        r["bertscore_human_src"] = bertscore_f1(human, src)
        r["bertscore_gen_human"] = bertscore_f1(gen, human)
    r["ibleu"] = IBLEU_ALPHA * r.bleu_gen_human - (1 - IBLEU_ALPHA) * r.bleu_gen_src
    r.to_csv(f"{VS_HUMAN_DIR}/{dataset}_scores.csv", index=False)

    return {"dataset": dataset, "n_label1": len(r),
            "BLEU_gen_src (Qwen rewording)":    round(corpus_bleu_score(gen, src), 2),
            "BLEU_human_src (human rewording)": round(corpus_bleu_score(human, src), 2),
            "BLEU_gen_human":                   round(corpus_bleu_score(gen, human), 2),
            "iBLEU": round(IBLEU_ALPHA * corpus_bleu_score(gen, human)
                           - (1 - IBLEU_ALPHA) * corpus_bleu_score(gen, src), 2),
            "BERTScore_gen_src":   round(r.bertscore_gen_src.mean(), 3),
            "BERTScore_human_src": round(r.bertscore_human_src.mean(), 3),
            "BERTScore_gen_human": round(r.bertscore_gen_human.mean(), 3)}

vs_human = pd.DataFrame([score_paraphrases_vs_human(ds) for ds in DATASETS])
vs_human.to_csv(f"{VS_HUMAN_DIR}/summary.csv", index=False)
print(vs_human.T.to_string())

                                      0      1      2
dataset                            mrpc    qqp   paws
n_label1                            279   1835   3536
BLEU_gen_src (Qwen rewording)     27.64  16.78  33.31
BLEU_human_src (human rewording)  48.21  29.26  65.49
BLEU_gen_human                    17.78   9.54  28.64
iBLEU                               8.7   4.27  16.25
BERTScore_gen_src                 0.674  0.656  0.665
BERTScore_human_src                0.71   0.67  0.866
BERTScore_gen_human               0.544  0.552  0.639


In [9]:
import json

nb = json.load(open("qwen_prob.ipynb"))
keys = ["def ", "logit", "prompt", "SYSTEM", "messages", "apply_chat_template", "not_paraphrase"]

for i, c in enumerate(nb["cells"]):
    if c["cell_type"] != "code":
        continue
    src = "".join(c["source"])
    if any(k in src for k in keys):
        print(f"\n===== cell {i} =====\n{src}")


===== cell 9 =====

SYSTEM = (
    "You are a paraphrase detector. Given two sentences, decide whether they "
    "express the same meaning. Answer 'paraphrase' only if all key facts and "
    "relations are preserved; otherwise 'not_paraphrase'. "
    'Respond only with valid JSON: {"label": "paraphrase"} or {"label": "not_paraphrase"}.'
)
PREFILL = '{"label": "'     # we stop here and read the model's next-token choice

def build(s1, s2):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Sentence 1: {s1}\nSentence 2: {s2}"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + PREFILL

def first_new_token(cand):
    base = tok(build("a", "b"), add_special_tokens=False).input_ids
    full = tok(build("a", "b") + cand, add_special_tokens=False).input_ids
    assert full[:len(base)] == base, "tokenization merged across the prefill boundary"
    return full[len(base)]

ID_PARA, ID_NOT = first_new_token("paraphr

In [10]:
# ===== Qwen as judge: does Qwen accept (source, Qwen paraphrase) as a paraphrase? =====
import os, torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_ID, QWEN_CACHE = "Qwen/Qwen2.5-7B-Instruct", "/scratch/bhandari.ru/hf_cache/hub"
IDENT_DIR = "qwen_hf_results"                       # your identification run on (sentence1, sentence2)

# --- identical prompt to qwen_prob.ipynb ---
JUDGE_SYSTEM = (
    "You are a paraphrase detector. Given two sentences, decide whether they "
    "express the same meaning. Answer 'paraphrase' only if all key facts and "
    "relations are preserved; otherwise 'not_paraphrase'. "
    'Respond only with valid JSON: {"label": "paraphrase"} or {"label": "not_paraphrase"}.'
)
PREFILL = '{"label": "'

def load_qwen():
    """Reuse the model already in memory (from generation); load it only if missing."""
    g = globals()
    if g.get("gen_model") is None:
        g["gtok"] = AutoTokenizer.from_pretrained(QWEN_ID, cache_dir=QWEN_CACHE, local_files_only=True,
                                                  padding_side="left")
        g["gen_model"] = AutoModelForCausalLM.from_pretrained(
            QWEN_ID, cache_dir=QWEN_CACHE, local_files_only=True,
            torch_dtype=torch.bfloat16, device_map="cuda").eval()
    assert gtok.padding_side == "left", "judge reads the last position, so padding must be on the left"
    print(f"Qwen ready | GPU memory {torch.cuda.memory_allocated()/1e9:.1f} GB")

def build_judge_prompt(s1, s2):
    msgs = [{"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": f"Sentence 1: {s1}\nSentence 2: {s2}"}]
    return gtok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True) + PREFILL

def first_label_token(word):
    base = gtok(build_judge_prompt("a", "b"), add_special_tokens=False).input_ids
    full = gtok(build_judge_prompt("a", "b") + word, add_special_tokens=False).input_ids
    assert full[:len(base)] == base, "tokenization merged across the prefill boundary"
    return full[len(base)]

@torch.no_grad()
def qwen_paraphrase_prob(s1_list, s2_list, bs=32):
    """P('paraphrase') vs 'not_paraphrase' at the label position, same as the identification run."""
    out = []
    for i in range(0, len(s1_list), bs):
        texts = [build_judge_prompt(a, b) for a, b in zip(s1_list[i:i+bs], s2_list[i:i+bs])]
        enc = gtok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        logits = gen_model(**enc).logits[:, -1, :].float()
        two = torch.stack([logits[:, ID_NOT], logits[:, ID_PARA]], dim=1)
        out.extend(torch.softmax(two, dim=1)[:, 1].cpu().tolist())
    return out

def check_matches_identification_run(n=256):
    """Re-score original PAWS pairs; should reproduce the saved identification probabilities."""
    saved = pd.read_csv(f"{IDENT_DIR}/qwen_hf_paws_results.csv").head(n)
    p = np.array(qwen_paraphrase_prob(saved.sentence1.tolist(), saved.sentence2.tolist()))
    agree = ((p >= 0.5).astype(int) == saved.pred.values).mean()
    print(f"sanity check on {n} PAWS pairs: label agreement {agree:.3f} | "
          f"max prob difference {np.abs(p - saved.qwen_logit_prob.values).max():.4f}")
    assert agree > 0.97, "judge does not match the identification run: stop and check"

def judge_generated_paraphrases(dataset, chunk=640):
    """Score (source, Qwen paraphrase) for every test row; resumes from a partial file."""
    final = f"{VS_SOURCE_DIR}/{dataset}_qwen_judge.csv"
    if os.path.exists(final):
        return pd.read_csv(final)
    gen = load_generated_paraphrases(dataset)
    src, para = gen.sentence1.map(clean_text).tolist(), gen.gen.map(clean_text).tolist()
    part = f"{VS_SOURCE_DIR}/{dataset}_qwen_judge_partial.csv"
    probs = pd.read_csv(part).qwen_judge_prob.tolist() if os.path.exists(part) else []
    print(f"{dataset}: {len(gen)} pairs, resuming from {len(probs)}")
    for i in range(len(probs), len(gen), chunk):
        j = min(i + chunk, len(gen))
        probs += qwen_paraphrase_prob(src[i:j], para[i:j])
        pd.DataFrame({"qwen_judge_prob": probs}).to_csv(part, index=False)
        print(f"  {j}/{len(gen)}")
    out = gen[["idx", "label"]].copy()
    out["qwen_judge_prob"] = probs
    out["qwen_judge_accept"] = (out.qwen_judge_prob >= 0.5).astype(int)
    out.to_csv(final, index=False); os.remove(part)
    return out

# --- run ---
load_qwen()
ID_PARA, ID_NOT = first_label_token("paraphrase"), first_label_token("not_paraphrase")
print("label tokens:", repr(gtok.decode([ID_PARA])), "|", repr(gtok.decode([ID_NOT])))
check_matches_identification_run()

judged = {ds: judge_generated_paraphrases(ds) for ds in DATASETS}

# summary 1: paraphrase vs source, all rows
rows = []
for ds, j in judged.items():
    lo, hi = bootstrap_ci95(j.qwen_judge_accept)
    rows.append({"dataset": ds, "n": len(j),
                 "acceptance_rate": round(j.qwen_judge_accept.mean(), 3), "acceptance_CI95": f"[{lo}, {hi}]",
                 "mean_prob": round(j.qwen_judge_prob.mean(), 3),
                 "confident_%(p<0.05 or >0.95)": round(100 * ((j.qwen_judge_prob < .05) | (j.qwen_judge_prob > .95)).mean(), 1)})
qwen_judge_vs_source = pd.DataFrame(rows)
qwen_judge_vs_source.to_csv(f"{VS_SOURCE_DIR}/qwen_judge_summary.csv", index=False)
print("\n=== Qwen judge: (source, Qwen paraphrase), all rows ===")
print(qwen_judge_vs_source.to_string(index=False))

# summary 2: human vs Qwen paraphrases, same label-1 rows
rows = []
for ds, j in judged.items():
    ident = pd.read_csv(f"{IDENT_DIR}/qwen_hf_{ds}_results.csv")
    assert len(ident) == len(j) and (ident.label.values == j.label.values).all(), f"{ds}: rows not aligned"
    m = (j.label == 1).values
    rows.append({"dataset": ds, "n_label1": int(m.sum()),
                 "accepts_human_paraphrase": round(ident.pred.values[m].mean(), 3),
                 "accepts_qwen_paraphrase":  round(j.qwen_judge_accept.values[m].mean(), 3),
                 "mean_prob_human": round(ident.qwen_logit_prob.values[m].mean(), 3),
                 "mean_prob_qwen":  round(j.qwen_judge_prob.values[m].mean(), 3)})
qwen_judge_human_vs_qwen = pd.DataFrame(rows)
qwen_judge_human_vs_qwen.to_csv(f"{VS_HUMAN_DIR}/qwen_judge_human_vs_qwen.csv", index=False)
print("\n=== Qwen judge: human vs Qwen paraphrases (label-1 rows) ===")
print(qwen_judge_human_vs_qwen.to_string(index=False))

2026-11-24 13:28:18.916138: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795544900.642313  703295 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795544901.027222  703295 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795544906.223929  703295 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795544906.223978  703295 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795544906.223980  703295 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Qwen ready | GPU memory 15.3 GB
label tokens: 'par' | 'not'
sanity check on 256 PAWS pairs: label agreement 1.000 | max prob difference 0.0000
mrpc: 408 pairs, resuming from 0
  408/408
qqp: 5000 pairs, resuming from 0
  640/5000
  1280/5000
  1920/5000
  2560/5000
  3200/5000
  3840/5000
  4480/5000
  5000/5000
paws: 8000 pairs, resuming from 0
  640/8000
  1280/8000
  1920/8000
  2560/8000
  3200/8000
  3840/8000
  4480/8000
  5120/8000
  5760/8000
  6400/8000
  7040/8000
  7680/8000
  8000/8000

=== Qwen judge: (source, Qwen paraphrase), all rows ===
dataset    n  acceptance_rate acceptance_CI95  mean_prob  confident_%(p<0.05 or >0.95)
   mrpc  408            0.953  [0.934, 0.971]      0.954                          99.0
    qqp 5000            0.970  [0.965, 0.974]      0.968                          98.4
   paws 8000            0.857  [0.849, 0.865]      0.855                          96.0

=== Qwen judge: human vs Qwen paraphrases (label-1 rows) ===
dataset  n_label1  accepts_hum

In [11]:
import textwrap

ds = "paws"
judge  = pd.read_csv(f"{VS_SOURCE_DIR}/{ds}_qwen_judge.csv")
scores = pd.read_csv(f"{VS_SOURCE_DIR}/{ds}_scores.csv")[["idx", "sentence1", "gen", "bertscore_gen_src", "bleu_gen_src"]]
d = judge.merge(scores, on="idx")

rejected = d[d.qwen_judge_accept == 0]
print(f"{ds}: {len(rejected)} of {len(d)} Qwen paraphrases rejected by Qwen ({len(rejected)/len(d):.1%})\n")

def show(rows, title):
    print("=" * 100 + f"\n{title}\n" + "=" * 100)
    for r in rows.itertuples():
        print(f"\n[idx {r.idx}] judge prob = {r.qwen_judge_prob:.3f} | BERTScore = {r.bertscore_gen_src:.3f} | BLEU = {r.bleu_gen_src:.1f}")
        print(textwrap.fill("SRC: " + r.sentence1, 98, subsequent_indent="     "))
        print(textwrap.fill("GEN: " + r.gen,       98, subsequent_indent="     "))

show(rejected.nsmallest(10, "qwen_judge_prob"), "10 most confident rejections")
show(d[(d.qwen_judge_prob > 0.3) & (d.qwen_judge_prob < 0.5)].sample(5, random_state=0), "5 borderline rejections (prob 0.3-0.5)")

paws: 1145 of 8000 Qwen paraphrases rejected by Qwen (14.3%)

10 most confident rejections

[idx 5233] judge prob = 0.000 | BERTScore = 0.278 | BLEU = 13.8
SRC: Born in Straja , in Austrian Bukovina , he attended high school in Suceava from 1897 to 1905
     .
GEN: In what year did he start attending high school in Suceava, and in which place was he born?

[idx 2231] judge prob = 0.000 | BERTScore = 0.250 | BLEU = 2.2
SRC: The station is owned by the Kenston Local School District , which serves Geauga County and
     eastern parts of Cleveland .
GEN: Which school district owns the station and what areas does it serve?

[idx 731] judge prob = 0.000 | BERTScore = 0.147 | BLEU = 2.5
SRC: Wilbur was born on 1 March 1921 in North Caldwell , New Jersey and grew up in New York City .
GEN: When was Wilbur born, and where did he spend his childhood?

[idx 1472] judge prob = 0.000 | BERTScore = 0.437 | BLEU = 6.7
SRC: In April 2014 , president Randy Garfield retired . Kenneth Svendsen was named 

In [12]:
def statement_to_question(sources, generations):
    src = pd.Series(list(sources)).astype(str).str.strip()
    gen = pd.Series(list(generations)).astype(str).str.strip()
    return ~src.str.endswith("?") & gen.str.endswith("?")

rows = []
for version, folder in [("unfixed", GEN_UNFIXED), ("fixed", GEN_FIXED)]:
    for ds in DATASETS:
        r = load_generated_paraphrases(ds, folder)
        s2q = statement_to_question(r.sentence1, r.gen)
        n_statements = (~r.sentence1.astype(str).str.strip().str.endswith("?")).sum()
        rows.append({"version": version, "dataset": ds, "statements": int(n_statements),
                     "statement_to_question": int(s2q.sum()),
                     "% of statements": round(100 * s2q.sum() / max(n_statements, 1), 2)})
print(pd.DataFrame(rows).to_string(index=False))

version dataset  statements  statement_to_question  % of statements
unfixed    mrpc         407                      0             0.00
unfixed     qqp         157                     57            36.31
unfixed    paws        8000                      2             0.02
  fixed    mrpc         407                    117            28.75
  fixed     qqp         157                     48            30.57
  fixed    paws        8000                   3987            49.84


In [4]:
# ===== Paraphrase generation v3: sentence-type-aware prompt (fixes statement->question) =====
import os, re, json, time, datetime, torch, transformers, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN_V3 = "qwen_hf_generation/taskA_v3"
os.makedirs(GEN_V3, exist_ok=True)
QWEN_ID, QWEN_CACHE = "Qwen/Qwen2.5-7B-Instruct", "/scratch/bhandari.ru/hf_cache/hub"
SEED, BATCH, SAVE_EVERY, MAX_RETRY = 42, 128, 5, 3
GEN_CFG = dict(temperature=0.7, top_p=0.9, top_k=40, max_new_tokens=128)

BASE_PROMPT = ("You are a precise paraphrasing assistant. Rewrite the given text using different wording "
               "and sentence structure while preserving the exact meaning. Keep all names, numbers, places, "
               "dates and relations unchanged. Do not add or remove information. Respond in English only. "
               "Output only the paraphrase, nothing else.")
TYPE_RULE = {"question":  " The input is a question: the paraphrase must also be a question. Do not answer it.",
             "statement": " The input is a statement: the paraphrase must also be a statement, not a question."}

QUESTION_START = re.compile(r"^(what|how|why|who|whom|whose|when|where|which|is|are|was|were|am|can|could|"
                            r"do|does|did|should|would|will|shall|may|might|has|have|had)\b", re.I)
CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")

def is_question(text):
    t = str(text).strip()
    return t.endswith("?") or bool(QUESTION_START.match(t))

def ensure_qwen_loaded():
    g = globals()
    if g.get("gen_model") is None:
        g["gtok"] = AutoTokenizer.from_pretrained(QWEN_ID, cache_dir=QWEN_CACHE, local_files_only=True, padding_side="left")
        g["gen_model"] = AutoModelForCausalLM.from_pretrained(QWEN_ID, cache_dir=QWEN_CACHE, local_files_only=True,
                                                              torch_dtype=torch.bfloat16, device_map="cuda").eval()
    if g.get("SUPPRESS") is None:                          # block every token containing CJK characters
        special = set(gtok.all_special_ids)
        g["SUPPRESS"] = [i for i in range(len(gtok)) if i not in special and CJK.search(gtok.decode([i]))]
    gtok.padding_side = "left"

def generate_paraphrases(sentences):
    prompts = []
    for s in sentences:
        system = BASE_PROMPT + TYPE_RULE["question" if is_question(s) else "statement"]
        prompts.append(gtok.apply_chat_template([{"role": "system", "content": system},
                                                 {"role": "user", "content": s}],
                                                tokenize=False, add_generation_prompt=True))
    enc = gtok(prompts, return_tensors="pt", padding=True).to("cuda")
    with torch.no_grad():
        out = gen_model.generate(**enc, do_sample=True, pad_token_id=gtok.pad_token_id,
                                 suppress_tokens=SUPPRESS, **GEN_CFG)
    new = out[:, enc.input_ids.shape[1]:]
    hit_limit = [(row != gtok.pad_token_id).sum().item() >= GEN_CFG["max_new_tokens"] for row in new]
    return [t.strip() for t in gtok.batch_decode(new, skip_special_tokens=True)], hit_limit

def generation_problems(sources, generations):
    src = pd.Series(list(sources)).astype(str).str.strip()
    gen = pd.Series(list(generations)).astype(str).str.strip()
    src_q = src.map(is_question)
    return pd.DataFrame({"flag_cjk": gen.str.contains(CJK) & ~src.str.contains(CJK),
                         "flag_type_flip": src_q != gen.str.endswith("?")})   # either direction

def run_paraphrase_generation_v3(dataset):
    final, part = f"{GEN_V3}/{dataset}_taskA.csv", f"{GEN_V3}/{dataset}_partial.csv"
    if os.path.exists(final):
        print(f"{dataset}: already done"); return pd.read_csv(final)
    df = pd.read_csv(f"splits/{dataset}_test.csv").reset_index().rename(columns={"index": "idx"})
    df["sentence1"] = df.sentence1.astype(str)
    done = pd.read_csv(part) if os.path.exists(part) else pd.DataFrame(columns=["idx", "gen", "truncated"])
    todo = df[~df.idx.isin(done.idx)]
    todo = todo.assign(_len=todo.sentence1.str.len()).sort_values(["_len", "idx"])
    b0, rows, t0 = len(done) // BATCH, [], time.time()
    print(f"{dataset}: {len(df)} rows | {len(done)} done | {len(todo)} to go")

    for i in range(0, len(todo), BATCH):
        chunk = todo.iloc[i:i + BATCH]
        torch.manual_seed(SEED + b0 + i // BATCH)
        gens, cut = generate_paraphrases(chunk.sentence1.tolist())
        rows += [{"idx": j, "gen": g, "truncated": c} for j, g, c in zip(chunk.idx, gens, cut)]
        if (i // BATCH + 1) % SAVE_EVERY == 0 or i + BATCH >= len(todo):
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); rows = []
            done.to_csv(part, index=False)
            print(f"  {len(done):5d}/{len(df)}  {(i + len(chunk)) / (time.time() - t0):5.1f} sent/s")

    done["idx"] = done.idx.astype(int)
    res = df.merge(done, on="idx", how="left").sort_values("idx").reset_index(drop=True)
    assert len(res) == len(df) and res.gen.notna().all(), f"{dataset}: missing generations"

    res["n_retries"] = 0
    for attempt in range(1, MAX_RETRY + 1):
        p = generation_problems(res.sentence1, res.gen); bad = (p.flag_cjk | p.flag_type_flip).values
        if not bad.any(): break
        torch.manual_seed(SEED + 100_000 + attempt)
        regen, cut = generate_paraphrases(res.sentence1[bad].tolist())
        res.loc[bad, "gen"], res.loc[bad, "truncated"], res.loc[bad, "n_retries"] = regen, cut, attempt
        print(f"  retry {attempt}: regenerated {bad.sum()} rows")

    p = generation_problems(res.sentence1, res.gen)
    res["src_is_question"] = res.sentence1.map(is_question)
    res["flag_cjk"], res["flag_type_flip"] = p.flag_cjk.values, p.flag_type_flip.values
    res = res[["idx", "sentence1", "sentence2", "label", "gen", "truncated", "n_retries",
               "src_is_question", "flag_cjk", "flag_type_flip"]]
    res.to_csv(final, index=False); os.remove(part)
    return res

# --- run ---
ensure_qwen_loaded()
for ds in ["mrpc", "qqp", "paws"]:
    s = pd.read_csv(f"splits/{ds}_test.csv").sentence1.astype(str)
    print(f"{ds}: sources treated as questions = {s.map(is_question).sum()} of {len(s)}")

gen_v3 = {ds: run_paraphrase_generation_v3(ds) for ds in ["mrpc", "qqp", "paws"]}

json.dump({"version": "v3: sentence-type-aware prompt + CJK block + retries (both flip directions)",
           "model": QWEN_ID, "dtype": "bfloat16", "transformers": transformers.__version__,
           "base_prompt": BASE_PROMPT, "type_rules": TYPE_RULE, "gen_config": GEN_CFG,
           "seed": SEED, "batch_size": BATCH, "max_retry": MAX_RETRY,
           "date": datetime.datetime.now().isoformat(timespec="seconds")},
          open(f"{GEN_V3}/metadata.json", "w"), indent=1)

print()
for ds, r in gen_v3.items():
    g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
    print(f"{ds:5s} n={len(r):5d} | type flips={int(r.flag_type_flip.sum()):3d} | CJK={int(r.flag_cjk.sum())} | "
          f"retried={(r.n_retries > 0).sum():3d} | empty={(g == '').sum()} | copies={(g.str.lower() == s.str.lower()).sum()} | "
          f"truncated={int(r.truncated.astype(bool).sum())} | len ratio={(g.str.len() / s.str.len()).median():.2f}")

2026-11-24 17:33:37.167545: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1795559617.533504 1445921 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1795559617.604868 1445921 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1795559618.488998 1445921 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795559618.489029 1445921 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1795559618.489031 1445921 computation_placer.cc:177] computation placer alr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

mrpc: sources treated as questions = 1 of 408
qqp: sources treated as questions = 4878 of 5000
paws: sources treated as questions = 52 of 8000
mrpc: already done
qqp: already done
paws: already done

mrpc  n=  408 | type flips=  0 | CJK=0 | retried=  0 | empty=0 | copies=0 | truncated=0 | len ratio=1.07
qqp   n= 5000 | type flips= 46 | CJK=0 | retried= 67 | empty=0 | copies=1 | truncated=0 | len ratio=1.24
paws  n= 8000 | type flips=  0 | CJK=0 | retried=  7 | empty=0 | copies=0 | truncated=0 | len ratio=1.05


In [14]:
# ===== Tighten the question rule, redo only the affected datasets =====
def is_question_old(text):
    t = str(text).strip()
    return t.endswith("?") or bool(QUESTION_START.match(t))

def is_question(text):            # replaces the old rule; generate_paraphrases() and generation_problems() use it
    t = str(text).strip()
    return t.endswith("?") or (bool(QUESTION_START.match(t)) and not t.endswith("."))

paws_src = pd.read_csv("splits/paws_test.csv").sentence1.astype(str)
print("PAWS sources the old rule called questions (first 6):")
for s in paws_src[paws_src.map(is_question_old)].head(6):
    print("  -", s[:110])

for ds in ["mrpc", "qqp", "paws"]:
    src = pd.read_csv(f"splits/{ds}_test.csv").sentence1.astype(str)
    old_q, new_q = src.map(is_question_old), src.map(is_question)
    changed = int((old_q != new_q).sum())
    print(f"{ds}: questions old={old_q.sum()} new={new_q.sum()} | changed={changed}")
    if changed:                                   # regenerate this dataset with the corrected rule
        for f in [f"{GEN_V3}/{ds}_taskA.csv", f"{GEN_V3}/{ds}_partial.csv"]:
            if os.path.exists(f):
                os.remove(f)

gen_v3 = {ds: run_paraphrase_generation_v3(ds) for ds in ["mrpc", "qqp", "paws"]}

print()
for ds, r in gen_v3.items():
    g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
    print(f"{ds:5s} n={len(r):5d} | questions in source={int(r.src_is_question.sum()):4d} | "
          f"type flips={int(r.flag_type_flip.sum()):3d} | CJK={int(r.flag_cjk.sum())} | "
          f"retried={(r.n_retries > 0).sum():3d} | empty={(g == '').sum()} | "
          f"copies={(g.str.lower() == s.str.lower()).sum()} | len ratio={(g.str.len() / s.str.len()).median():.2f}")

PAWS sources the old rule called questions (first 6):
  - Will and Dani sleep together , but he returns to Gypsy and supports her after being raped by Sam Atwell ( Kane
  - When he died in 1719 , his eldest son , also called Joseph , followed initially Joseph Truman .
  - Where is invariant mass and invariant energy equals . The squared mass is still stable and the particle can be
  - When Twain heard of the writer who was called `` the Jewish Mark Twain '' , he replied `` please tell him that
  - Will and Gypsy sleep together , but he returns to Dani and supports her after being raped by Kane Phillips ( S
  - Will and Gypsy sleep together but he returns to Dani and supports her after she is raped by Kane Phillips ( Sa
mrpc: questions old=1 new=1 | changed=0
qqp: questions old=4878 new=4863 | changed=15
paws: questions old=52 new=7 | changed=45
mrpc: already done
qqp: 5000 rows | 0 done | 5000 to go
    640/5000   63.6 sent/s
   1280/5000   66.8 sent/s
   1920/5000   66.0 sent/s
   256

In [15]:
# ===== Patch: in PAWS/MRPC only a "?" makes a sentence a question; redo misclassified rows =====
def is_question_strict(text):
    return str(text).strip().endswith("?")

for ds in ["mrpc", "paws"]:
    path = f"{GEN_V3}/{ds}_taskA.csv"
    r = pd.read_csv(path)
    fix = (r.src_is_question & ~r.sentence1.astype(str).str.strip().str.endswith("?")).values
    print(f"{ds}: {fix.sum()} rows to regenerate as statements")
    if not fix.any():
        continue

    rule_in_use = is_question
    is_question = is_question_strict                 # generate_paraphrases() now picks the statement rule
    for attempt in range(3):                         # retry if any still comes out as a question
        torch.manual_seed(SEED + 200_000 + attempt)
        todo = fix & (r.src_is_question.values | r.gen.astype(str).str.strip().str.endswith("?").values)
        if not todo.any():
            break
        new, cut = generate_paraphrases(r.sentence1[todo].tolist())
        r.loc[todo, "gen"], r.loc[todo, "truncated"] = new, cut
        r.loc[todo, "src_is_question"] = False
        r.loc[todo, "n_retries"] = r.loc[todo, "n_retries"] + 1
    is_question = rule_in_use

    g = r.gen.astype(str).str.strip()
    r.loc[fix, "flag_type_flip"] = g[fix].str.endswith("?").values
    r.loc[fix, "flag_cjk"] = g[fix].str.contains(CJK).values
    r.to_csv(path, index=False)
    gen_v3[ds] = r
    for s, gnew in zip(r.sentence1[fix], g[fix]):
        print(f"   SRC: {s[:100]}\n   GEN: {gnew[:100]}\n")

print("final type flips:", {ds: int(r.flag_type_flip.sum()) for ds, r in gen_v3.items()})

mrpc: 0 rows to regenerate as statements
paws: 7 rows to regenerate as statements
   SRC: is the national member federation for Grenada in the following international organisations :
   GEN: Grenada's national member federation is part of the following international organizations:

   SRC: When Mark Twain heard of the writer called `` the Jewish Sholem Aleichem , '' he replied , `` please
   GEN: Upon learning about the writer known as "the Jewish Sholem Aleichem," Mark Twain responded by saying

   SRC: is the national member federation for Grenada in the following international organisations :
   GEN: Grenada's national member federation is part of the following international organizations:

   SRC: When Mark Twain heard of the writer called `` the Jewish Sholem Aleichem , '' he replied , `` please
   GEN: When Mark Twain learned about the writer known as "the Jewish Sholem Aleichem," he responded by sayi

   SRC: is the international member federation for Grenada in the following na

In [2]:
import requests, time, pandas as pd
r = gen_v3["qqp"] if "gen_v3" in globals() else pd.read_csv("qwen_hf_generation/taskA_v3/qqp_taskA.csv")
flips = r[r.flag_type_flip]
print(f"QQP type flips: {len(flips)} | source ends with '?': {flips.sentence1.str.strip().str.endswith('?').sum()}")
for x in flips.sample(min(12, len(flips)), random_state=0).itertuples():
    print(f"\n[{x.idx}] SRC: {x.sentence1[:120]}\n       GEN: {x.gen[:120]}")

/home/bhandari.ru/.local/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (7.4.0.post1)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


QQP type flips: 46 | source ends with '?': 1

[2952] SRC: What are my chances to get into IVY League? Upenn particularly.
       GEN: What are my chances of getting into the UPenn Ivy League school?

[2792] SRC: "Whence the term ""shitshow?"""
       GEN: Where did the term "shitshow" originate?

[3365] SRC: "What does it mean when girls call a guy ""cute"", is it different to “hot”?"
       GEN: When girls refer to a guy as "cute," what does that signify, and how does it differ from calling him "hot"?

[552] SRC: What is your deepest secret that you have never told anyone in real life ? (submit anonymously)
       GEN: Do you have any deeply hidden secret that you've kept to yourself without sharing with anyone personally? (Please submit

[1428] SRC: What should i do know as i broke up with my gf and i lives in hostel and my roommate has gone to his town. I want to sui
       GEN: What steps should I take now that I've broken up with my girlfriend, am living in a hostel where my roomm

In [5]:
# ===== QQP: better question rule (quotes, text after "?"), redo only rows told the wrong type =====
def is_question_qqp(text):
    t = str(text).strip().strip('"').strip()
    return ("?" in t) or (bool(QUESTION_START.match(t)) and not t.endswith("."))

path = f"{GEN_V3}/qqp_taskA.csv"
r = pd.read_csv(path)
src_q_new = r.sentence1.map(is_question_qqp)
gen_q = r.gen.astype(str).str.strip().str.endswith("?")

told_wrong   = (src_q_new != r.src_is_question)                 # rows where the prompt got the wrong type
hidden_wrong = (told_wrong & src_q_new & ~gen_q).values         # real question, told "statement", came out a statement
print(f"QQP: rows told the wrong type = {told_wrong.sum()} | "
      f"of those, Qwen wrote a statement for a real question = {hidden_wrong.sum()}")

if hidden_wrong.any():
    rule_in_use = is_question
    is_question = is_question_qqp                                # generate_paraphrases() now gives the question rule
    for attempt in range(3):
        todo = hidden_wrong & ~r.gen.astype(str).str.strip().str.endswith("?").values
        if not todo.any():
            break
        torch.manual_seed(SEED + 300_000 + attempt)
        new, cut = generate_paraphrases(r.sentence1[todo].tolist())
        r.loc[todo, "gen"], r.loc[todo, "truncated"] = new, cut
        r.loc[todo, "n_retries"] = r.loc[todo, "n_retries"] + 1
    is_question = rule_in_use
    for s, g in zip(r.sentence1[hidden_wrong][:5], r.gen[hidden_wrong][:5]):
        print(f"   SRC: {s[:110]}\n   GEN: {g[:110]}\n")

# recompute flags with the better rule and save
r["src_is_question"] = r.sentence1.map(is_question_qqp)
r["flag_type_flip"] = r.src_is_question != r.gen.astype(str).str.strip().str.endswith("?")
r.to_csv(path, index=False)
gen_v3["qqp"] = r
print("QQP type flips with the better rule:", int(r.flag_type_flip.sum()))
for x in r[r.flag_type_flip].head(5).itertuples():
    print(f"  [{x.idx}] SRC: {x.sentence1[:100]}\n          GEN: {x.gen[:100]}")

QQP: rows told the wrong type = 137 | of those, Qwen wrote a statement for a real question = 96
   SRC: "If a girl texts ""good night, sweet dreams"" to me but puts a heart emoji after it does it mean anything or i
   GEN: "If a girl sends 'good night, sweet dreams' followed by a heart emoji, is there any underlying meaning beyond 

   SRC: "If 110/70 is normal blood pressure, what is the normal range for the ""low number"" for a young female adult?
   GEN: "If 110/70 is considered normal blood pressure, what is the typical normal range for the lower number in young

   SRC: South American warrior tribes vs European Viking warriors [men & women], which one will win? PART TWO View Cha
   GEN: Which tribe would emerge victorious, the South American warrior tribes or the European Viking warriors [includ

   SRC: "What are some other ways to say ""as mentioned previously"" in a formal text?"
   GEN: "What are some alternative phrases for ""as mentioned previously"" in formal writing?"

   

In [7]:
# ===== QQP: check sentence type after removing CSV-style wrapping quotes (no regeneration) =====
def strip_wrapping_quotes(text):
    t = str(text).strip().replace('""', '"')          # CSV-escaped quotes -> normal quotes
    return t.strip('"').strip("'").strip()

path = f"{GEN_V3}/qqp_taskA.csv"
r = pd.read_csv(path)
gen_is_question = r.gen.map(lambda t: strip_wrapping_quotes(t).endswith("?"))
r["flag_type_flip"] = r.src_is_question != gen_is_question
r.to_csv(path, index=False)
gen_v3["qqp"] = r

n_wrapped = r.gen.astype(str).str.strip().str.startswith('"').sum()
print(f"QQP outputs wrapped in quotes: {n_wrapped}")
print(f"QQP type flips after stripping quotes: {int(r.flag_type_flip.sum())}")
for x in r[r.flag_type_flip].head(8).itertuples():
    print(f"  [{x.idx}] SRC: {x.sentence1[:100]}\n          GEN: {x.gen[:100]}")

# record the final rules in the v3 metadata
meta_path = f"{GEN_V3}/metadata.json"
meta = json.load(open(meta_path))
meta["question_rule"] = {
    "mrpc, paws": "question only if the sentence ends with '?' (both are statement datasets)",
    "qqp": "question if '?' appears anywhere after removing wrapping quotes, or it starts with a question word "
           "and does not end with '.'",
    "output check": "wrapping quotes removed before checking for a final '?'"}
json.dump(meta, open(meta_path, "w"), indent=1)
print("\nmetadata updated:", meta_path)

QQP outputs wrapped in quotes: 47
QQP type flips after stripping quotes: 11
  [552] SRC: What is your deepest secret that you have never told anyone in real life ? (submit anonymously)
          GEN: Do you have any deeply hidden secret that you've kept to yourself without sharing with anyone person
  [1428] SRC: What should i do know as i broke up with my gf and i lives in hostel and my roommate has gone to his
          GEN: What steps should I take now that I've broken up with my girlfriend, am living in a hostel where my 
  [1773] SRC: Anybody willing to do online group study for cfa level 1 exam 2015? Let me know
          GEN: Is anyone interested in participating in an online group study session for the CFA Level 1 exam from
  [2140] SRC: "What is the output of following code? And then why? Printf (""%c"", 219);"
          GEN: What would be the result of this code snippet, and can you explain why? `printf ("%c", 219);`
  [2300] SRC: How can I get unlimited Ola Credits? Please h

In [8]:
DATASETS      = ["mrpc", "qqp", "paws"]
GEN_UNFIXED   = "qwen_hf_generation/taskA_v1_unfixed"   # original prompt
GEN_V2        = "qwen_hf_generation/taskA_v2"           # over-corrected prompt (statement -> question bug)
GEN_FIXED     = "qwen_hf_generation/taskA_v3"           # final: sentence-type-aware prompt
METRICS_DIR   = "qwen_hf_generation/metrics_v3"         # fresh folder, so no v2 scores can be reused
VS_SOURCE_DIR = f"{METRICS_DIR}/vs_source"
VS_HUMAN_DIR  = f"{METRICS_DIR}/vs_human_reference"
for d in (VS_SOURCE_DIR, VS_HUMAN_DIR):
    os.makedirs(d, exist_ok=True)
IBLEU_ALPHA = 0.8

In [9]:
def clean_text(t):
    """Same cleanup for source, human reference and generation, so formatting differences don't count."""
    t = str(t).strip().replace('""', '"')                     # CSV-escaped quotes -> normal quotes
    if len(t) > 1 and t[0] == t[-1] == '"':                    # drop quotes wrapping the whole text
        t = t[1:-1].strip()
    t = (t.replace("\u2018", "'").replace("\u2019", "'").replace("\u201c", '"').replace("\u201d", '"')
          .replace("\u2014", " - ").replace("\u2013", "-"))
    t = re.sub(r"\s+([,.;:!?%)\]])", r"\1", t)          # "2011 ," -> "2011,"
    t = re.sub(r"([(\[])\s+", r"\1", t)                   # "( born" -> "(born"
    t = re.sub(r"\s+'(s|re|ve|ll|d|m)\b", r"'\1", t)       # "John 's" -> "John's"
    t = re.sub(r"\s+n't\b", "n't", t)
    return re.sub(r"\s+", " ", t).strip()

In [10]:
def load_generated_paraphrases(dataset, folder=None):
    return pd.read_csv(f"{folder or GEN_FIXED}/{dataset}_taskA.csv")

def load_cached_scores(dataset, generated):
    """Reuse per-row scores only if the file exists and both row IDs and generated text match."""
    path = f"{METRICS_DIR}/{dataset}_auto_metrics.csv"
    if not os.path.exists(path):
        return None
    cached = pd.read_csv(path)
    aligned = (len(cached) == len(generated) and (cached.idx.values == generated.idx.values).all()
               and (cached.gen.astype(str).values == generated.gen.astype(str).values).all())
    return cached if aligned else None

In [11]:
# ===== Generation failures across prompt versions =====
CJK = re.compile(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af\uff00-\uffef]")
QUESTION_START = re.compile(r"^(what|how|why|who|whom|whose|when|where|which|is|are|was|were|am|can|could|"
                            r"do|does|did|should|would|will|shall|may|might|has|have|had)\b", re.I)

def unwrap(text):
    return str(text).strip().replace('""', '"').strip('"').strip("'").strip()

def source_is_question(dataset, text):
    t = unwrap(text)
    if dataset == "qqp":
        return ("?" in t) or (bool(QUESTION_START.match(t)) and not t.endswith("."))
    return t.endswith("?")                                     # MRPC and PAWS are statement datasets

def detect_generation_failures(dataset, sources, generations):
    src_q = pd.Series([source_is_question(dataset, s) for s in sources])
    gen_q = pd.Series([unwrap(g).endswith("?") for g in generations])
    src_cjk = pd.Series(list(sources)).astype(str).str.contains(CJK)
    gen_cjk = pd.Series(list(generations)).astype(str).str.contains(CJK)
    return pd.DataFrame({"chinese_text": gen_cjk & ~src_cjk,
                         "statement_to_question": ~src_q & gen_q,
                         "question_to_statement": src_q & ~gen_q})

def compare_prompt_versions():
    rows = []
    for version, folder in [("v1 original", GEN_UNFIXED), ("v2 over-corrected", GEN_V2), ("v3 final", GEN_FIXED)]:
        for ds in DATASETS:
            r = load_generated_paraphrases(ds, folder)
            f = detect_generation_failures(ds, r.sentence1, r.gen)
            g, s = r.gen.astype(str).str.strip(), r.sentence1.astype(str).str.strip()
            rows.append({"version": version, "dataset": ds, "n": len(r),
                         "chinese_text": int(f.chinese_text.sum()),
                         "statement_to_question": int(f.statement_to_question.sum()),
                         "question_to_statement": int(f.question_to_statement.sum()),
                         "flagged_%": round(100 * f.any(axis=1).mean(), 2),
                         "len_ratio_median": round((g.str.len() / s.str.len()).median(), 2)})
    return pd.DataFrame(rows)

failures = compare_prompt_versions()
failures.to_csv(f"{METRICS_DIR}/generation_failures_by_version.csv", index=False)
print(failures.to_string(index=False))

          version dataset    n  chinese_text  statement_to_question  question_to_statement  flagged_%  len_ratio_median
      v1 original    mrpc  408             4                      0                      0       0.98              1.06
      v1 original     qqp 5000            16                      0                     59       1.34              1.25
      v1 original    paws 8000            36                      2                      0       0.48              1.04
v2 over-corrected    mrpc  408             0                    119                      0      29.17              1.07
v2 over-corrected     qqp 5000             0                      0                     17       0.34              1.23
v2 over-corrected    paws 8000             0                   3993                      0      49.91              1.06
         v3 final    mrpc  408             0                      0                      0       0.00              1.07
         v3 final     qqp 5000          

In [12]:
r = load_generated_paraphrases("paws"); f = detect_generation_failures("paws", r.sentence1, r.gen)
print(r[f.statement_to_question.values][["idx", "sentence1", "gen"]].to_string(max_colwidth=120))

       idx                                                                                                                sentence1                                                                                                                      gen
3839  3839  Christopher Llewellyn ( born Christopher Griffiths ) is an actor known for his role as Logan in the film 'Where Were...  Christopher Llewellyn, whose birth name is Christopher Griffiths, is recognized for playing the character Logan in t...


In [14]:
# ===== Final evaluation: paraphrase vs source (every test row) =====
def score_paraphrases_vs_source(dataset, folder=None, out_dir=None):
    """BERTScore (meaning kept) and BLEU (words reused) of Qwen's paraphrase against the source sentence."""
    out_dir = out_dir or VS_SOURCE_DIR
    path = f"{out_dir}/{dataset}_scores.csv"
    generated = load_generated_paraphrases(dataset, folder)
    src, gen = generated.sentence1.map(clean_text), generated.gen.map(clean_text)

    saved = pd.read_csv(path) if os.path.exists(path) else None
    if saved is not None and (saved.gen.astype(str).values == generated.gen.astype(str).values).all():
        r = saved                                               # same generations: reuse
    else:
        r = generated[["idx", "sentence1", "gen", "label"]].copy()
        r["bleu_gen_src"]      = sentence_bleu_scores(gen, src)
        r["bertscore_gen_src"] = bertscore_f1(gen, src)
        r.to_csv(path, index=False)

    lo, hi = bootstrap_ci95(r.bertscore_gen_src)
    return {"dataset": dataset, "n": len(r),
            "BERTScore_gen_src": round(r.bertscore_gen_src.mean(), 3), "BERTScore_CI95": f"[{lo}, {hi}]",
            "BLEU_gen_src": round(corpus_bleu_score(gen, src), 2)}

vs_source = pd.DataFrame([score_paraphrases_vs_source(ds) for ds in DATASETS])
vs_source.to_csv(f"{VS_SOURCE_DIR}/summary.csv", index=False)
print(vs_source.to_string(index=False))

FileNotFoundError: [Errno 2] No such file or directory: 'None/mrpc_taskA.csv'